In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor
import numpy as np
import pandas as pd

from sklearn.linear_model import Ridge
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.impute import SimpleImputer
from catboost import CatBoostRegressor
try:
    from lightgbm import LGBMRegressor
except OSError as e:
    if "libomp" in str(e).lower():
        raise OSError(
            "LightGBM could not load libomp (OpenMP).\n"
            "macOS + Homebrew fix:\n"
            "  brew install libomp\n"
            "Then restart the kernel.\n"
            "Conda fix:\n"
            "  conda install -c conda-forge llvm-openmp\n"
        ) from e
    raise

BASE_DIR = Path("/Users/vishalsankarram/dsci558-project/game_feature_export/artifacts/no_sentiment_with_value")
parquet_path = BASE_DIR / "cleaned_data.parquet"

if not parquet_path.exists():
    raise FileNotFoundError(f"Missing file: {parquet_path}")

In [2]:
df = pd.read_parquet(parquet_path)
print("Original shape:", df.shape)

# Drop columns by prefix only
prefix_re = r"(mech_|cat_|price_|rank)"
df = df.loc[:, ~df.columns.str.contains(prefix_re, regex=True)]

# Drop known unnecessary columns if present
cols_to_drop = [
    "categories", "mechanisms", "primary_category", "game_name",
    "is_expansion", "has_description_embedding",
    "log1p_last_mean", "n_weeks_observed", "log1p_last_min", "log1p_last_max",
    "last_week_spread", "log1p_mean_hist", "log1p_median_weekly_mean",
    "mean_vs_last_ratio", "pct_change_first_to_last", "max_drawdown_mean",
    "weeks_since_last_obs", "calendar_span_weeks", "trailing_null_frac_12w",
    "mean_intraweek_range_last", "mean_intraweek_range_avg", "has_games_csv",
    "stage_c_split"
]
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

print("After drop:", df.shape)

Original shape: (21690, 350)
After drop: (21690, 31)


/var/folders/tq/jtgfkbln5bv4lmlbq2tvnq1h0000gn/T/ipykernel_95700/4092145206.py:6: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df = df.loc[:, ~df.columns.str.contains(prefix_re, regex=True)]


In [3]:
SHARE_COLS = ["coll_share_owns", "coll_share_wants", "coll_share_wtb", "coll_share_wtt"]
COUNT_COLS = ["coll_count_owns", "coll_count_wants", "coll_count_wtb", "coll_count_wtt"]

if all(c in df.columns for c in SHARE_COLS):
    TARGET_COLS = SHARE_COLS
elif all(c in df.columns for c in COUNT_COLS):
    TARGET_COLS = COUNT_COLS
else:
    raise ValueError(f"Missing targets. Need either {SHARE_COLS} or {COUNT_COLS}")

print("Targets:", TARGET_COLS)

Targets: ['coll_share_owns', 'coll_share_wants', 'coll_share_wtb', 'coll_share_wtt']


In [4]:
ID_AND_STRING_COLS = {
    "bgg_id",
    "game_name",
    "categories",
    "mechanisms",
    "primary_category",
    "stage_a_split",
    # "stage_c_split",
}

EMBED_COLS = ["mean_embedding", "description_embedding"]

def get_tabular_cols(frame: pd.DataFrame, target_cols: list[str]) -> list[str]:
    exclude = set(ID_AND_STRING_COLS) | set(target_cols) | set(EMBED_COLS)
    cols = []
    for c in frame.columns:
        if c in exclude:
            continue
        if pd.api.types.is_numeric_dtype(frame[c]):
            cols.append(c)
    return cols

tabular_cols = get_tabular_cols(df, TARGET_COLS)
print("Tabular columns:", len(tabular_cols))

def stack_embeddings(frame: pd.DataFrame, col: str) -> np.ndarray | None:
    if col not in frame.columns:
        return None
    arrs = [np.asarray(x, dtype=np.float32) for x in frame[col].values]
    return np.vstack(arrs)

def build_X(frame: pd.DataFrame, tabular_cols: list[str]) -> np.ndarray:
    parts = [frame[tabular_cols].to_numpy(dtype=np.float32)]
    for col in EMBED_COLS:
        emb = stack_embeddings(frame, col)
        if emb is not None:
            parts.append(emb)
    return np.hstack(parts)

# Keep only rows with valid targets
df_work = df.loc[np.isfinite(df[TARGET_COLS]).all(axis=1)].copy()
ids = df_work["bgg_id"].copy()
X = build_X(df_work, tabular_cols)
y = df_work[TARGET_COLS].to_numpy(dtype=np.float32)

print("X shape:", X.shape)
print("y shape:", y.shape)

Tabular columns: 23
X shape: (20626, 791)
y shape: (20626, 4)


In [5]:
def evaluate_model(model, X_test, y_test, target_cols, id_series, model_name, clip_min=0.0, clip_max=1.0):
    y_pred = model.predict(X_test)
    y_pred = np.asarray(y_pred)

    # Keep predictions in a sensible range for share-like targets
    y_pred = np.clip(y_pred, clip_min, clip_max)

    rows = []
    for i, col in enumerate(target_cols):
        yt = y_test[:, i]
        yp = y_pred[:, i]

        rows.append({
            "model": model_name,
            "target": col,
            "mae": mean_absolute_error(yt, yp),
            "rmse": np.sqrt(mean_squared_error(yt, yp)),
            "r2": r2_score(yt, yp),
        })

    rows.append({
        "model": model_name,
        "target": "overall",
        "mae": mean_absolute_error(y_test, y_pred),
        "rmse": np.sqrt(mean_squared_error(y_test, y_pred)),
        "r2": r2_score(y_test, y_pred, multioutput="uniform_average"),
    })

    metrics_df = pd.DataFrame(rows)

    pred_df = pd.DataFrame(y_pred, columns=[f"pred_{c}" for c in target_cols])
    # true_df = pd.DataFrame(y_test, columns=[f"true_{c}" for c in target_cols])

    # out_df = pd.concat([true_df, pred_df], axis=1)
    out_df = pd.concat([ pred_df], axis=1)
    out_df["bgg_id"] = id_series.values
    out_df["model"] = model_name

    return metrics_df, out_df

In [6]:
import joblib
MODELS_DIR = Path("/Users/vishalsankarram/dsci558-project/models")
lgbm_model = joblib.load(MODELS_DIR / "lightgbm_model.joblib")
# rf_model = joblib.load(MODELS_DIR / "random_forest_model.joblib")

metadata = joblib.load(MODELS_DIR / "metadata.joblib")

In [7]:
lgbm_metrics_df, lgbm_pred_df = evaluate_model(
    lgbm_model, X, y, TARGET_COLS, ids, model_name="LightGBM"
)

/opt/homebrew/Caskroom/miniconda/base/envs/dsci-558-project/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/dsci-558-project/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/dsci-558-project/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/dsci-558-project/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [8]:
print(lgbm_pred_df.head())
print(len(lgbm_pred_df))

   pred_coll_share_owns  pred_coll_share_wants  pred_coll_share_wtb  \
0              0.872889               0.042967             0.024973   
1              0.911287               0.028484             0.011983   
2              0.765894               0.004560             0.000000   
3              0.773494               0.052545             0.040193   
4              0.825044               0.039603             0.039849   

   pred_coll_share_wtt  bgg_id     model  
0             0.046851       1  LightGBM  
1             0.058681      10  LightGBM  
2             0.254412   10007  LightGBM  
3             0.133280  100172  LightGBM  
4             0.095685  100275  LightGBM  
20626


In [9]:
df = pd.read_parquet(parquet_path)
print("Original shape:", df.shape)
print(df.head())

Original shape: (21690, 350)
   bgg_id                                     mean_embedding  \
0       1  [-0.03231392055749893, 0.030320700258016586, 0...   
1      10  [-0.007255558390170336, 0.03381650522351265, -...   
2  100046  [-0.13280192017555237, -0.06126563996076584, 0...   
3   10007  [-0.03585081920027733, 0.03831823170185089, -0...   
4  100172  [-0.027283532544970512, 0.03955841809511185, -...   

                               description_embedding  n_reviews  \
0  [0.016889318823814392, 0.0013508893316611648, ...   4.000000   
1  [0.12207939475774765, 0.027984006330370903, -0...   4.000000   
2  [-0.02486669272184372, -0.02881644479930401, -...  -0.307692   
3  [-0.09014654904603958, -0.019658751785755157, ...  -0.253846   
4  [0.007031145039945841, 0.026811618357896805, -...   0.223077   

   review_count_feature  dispersion  median_distance_to_mean  \
0              1.482218    0.055270                -0.200072   
1              1.508356    0.204980                 0.0

In [10]:
# Drop columns by prefix only
prefix_re = r"(mech_|cat_|rank)"
df = df.loc[:, ~df.columns.str.contains(prefix_re, regex=True)]

/var/folders/tq/jtgfkbln5bv4lmlbq2tvnq1h0000gn/T/ipykernel_95700/1376397886.py:3: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df = df.loc[:, ~df.columns.str.contains(prefix_re, regex=True)]


In [11]:
# Drop known unnecessary columns if present
cols_to_drop = [
    "categories", "mechanisms", "primary_category", "game_name", "mean_embedding", "description_embedding",
    "is_expansion", "has_description_embedding", "has_games_csv",
    "stage_c_split",'n_reviews', 'review_count_feature', 'dispersion',
       'median_distance_to_mean', 'max_distance_to_mean', 'n_unique_reviewers',
       'reviews_per_user_mean','reviews_per_user_std', 'value_score_mean', 'value_score_std',
       'year', 'geek_rating', 'avg_rating', 'bayes_average', 'num_voters',
       'min_players', 'max_players', 'best_min_players', 'best_max_players',
       'min_playtime', 'max_playtime', 'min_age', 'complexity', 'stage_a_split'
]
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

print("After drop:", df.shape)

After drop: (21690, 27)


In [12]:
cols_to_merge = ["bgg_id"] + [f"pred_{c}" for c in TARGET_COLS]

df_final = (
    df.merge(lgbm_pred_df[cols_to_merge], on="bgg_id", how="inner")
      .drop(columns=TARGET_COLS)
      .rename(columns={f"pred_{c}": c for c in TARGET_COLS})
)

In [13]:
print(len(df_final))
print(df_final.shape)

20626
(20626, 27)


In [14]:
print(df_final.head())
ids = df_final["bgg_id"].copy()
df_final.drop(columns=["bgg_id"], inplace=True)
print(df_final.shape)
print(ids.shape)

   bgg_id  log1p_last_mean  n_weeks_observed  price_slope_4w  price_vol  \
0       1         0.285581          0.228916             0.1   4.000000   
1      10         0.349009         -0.325301             0.0  -0.329732   
2   10007        -0.057413          0.349398             0.0   0.133796   
3  100172         0.057999          0.789157             0.0   2.273990   
4  100275         0.105747          1.307229             0.0   2.522334   

   price_coverage  log1p_last_min  log1p_last_max  last_week_spread  \
0             0.0        0.304923        0.308309               4.0   
1            -1.0        0.384580        0.346328               0.0   
2             0.0       -0.030898       -0.057571               0.0   
3             0.0        0.087086        0.057125               0.0   
4             0.0        0.135898        0.104576               0.0   

   log1p_mean_hist  ...  max_drawdown_mean  weeks_since_last_obs  \
0         0.467945  ...           1.884206            

In [15]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

TARGET_COL = "avg_quality"

# use all columns except target and id as features
FEATURE_COLS = [c for c in df.columns if c not in [TARGET_COL, "bgg_id"]]

X = df_final[FEATURE_COLS]

In [16]:
def evaluate_model(model, X_test, id_series, clip_min=0.0, clip_max=1.0):
    y_pred = model.predict(X_test)
    y_pred = np.asarray(y_pred).ravel()
    print(y_pred.shape)
    # keep predictions in a sensible range
    y_pred = np.clip(y_pred, clip_min, clip_max)

    out_df = pd.DataFrame({
        "bgg_id": id_series.values,
        "pred_avg_quality": y_pred,
    })

    return out_df

In [17]:
MODELS_DIR = Path("/Users/vishalsankarram/dsci558-project/value_models")
rf_model = joblib.load(MODELS_DIR / "random_forest_model.joblib")

In [18]:
print(X.shape)
print(ids.shape)

(20626, 26)
(20626,)


In [19]:
rf_pred_df = evaluate_model(
    rf_model, X, ids
)

(20626,)


In [20]:
print(rf_pred_df.head())
print(len(rf_pred_df))

   bgg_id  pred_avg_quality
0       1          0.666448
1      10          0.711645
2   10007          0.580688
3  100172          0.625265
4  100275          0.650094
20626


In [21]:
rf_pred_df

,bgg_id,pred_avg_quality
0,1,0.666448
1,10,0.711645
2,10007,0.580688
3,100172,0.625265
4,100275,0.650094
...,...,...
20621,99875,0.710938
20622,99880,0.587672
20623,999,0.586581
20624,99935,0.694061


In [22]:
import os
import json
import glob
import numpy as np
import pandas as pd

# paths
mapping_csv = "bgo_key_bgg_map.tsv"      # your file with bgg_id <-> key
json_dir = "price_histories"            # folder containing key.json files
out_csv = "bgg_with_price_stats.csv"

# read mapping file
df_map = pd.read_csv(mapping_csv, sep=None, engine="python")  # works for csv or tsv
# if you know it's tab-separated, use: pd.read_csv(mapping_csv, sep="\t")

def extract_price_stats(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        obj = json.load(f)

    rows = []
    for block in obj.get("price_history", []):
        data = block.get("result", {}).get("data", [])
        rows.extend(data)

    if not rows:
        return {
            "mean_of_mean": np.nan,
            "max_of_max": np.nan,
            "min_of_min": np.nan,
            "n_points": 0,
        }

    means = [r.get("mean") for r in rows if r.get("mean") is not None]
    maxes = [r.get("max") for r in rows if r.get("max") is not None]
    mins  = [r.get("min") for r in rows if r.get("min") is not None]

    return {
        "mean_of_mean": float(np.mean(means)) if means else np.nan,
        "max_of_max": float(np.max(maxes)) if maxes else np.nan,
        "min_of_min": float(np.min(mins)) if mins else np.nan,
        "n_points": len(rows),
    }

# build stats dataframe from all key.json files
stats = []
for json_path in glob.glob(os.path.join(json_dir, "*.json")):
    key = os.path.splitext(os.path.basename(json_path))[0]
    row = {"key": key}
    row.update(extract_price_stats(json_path))
    stats.append(row)

df_stats = pd.DataFrame(stats)

# merge stats onto the mapping file
df_out = df_map.merge(df_stats, on="key", how="left")

print(df_out.head())

          key                                     slug  \
0  x5mxZxBV6L                               blood-rage   
1  7X4Ik831tA                                kanban-ev   
2  VbGKQCNCsI                                 agricola   
3  qJ2Bj1AmNu  the-castles-of-burgundy-special-edition   
4  46AnulxktA                                harmonies   

                                      title    bgg_id  \
0                                Blood Rage  170216.0   
1                                 Kanban EV  284378.0   
2                                  Agricola   31260.0   
3  The Castles of Burgundy: Special Edition  363622.0   
4                                 Harmonies  414317.0   

                                          bgg_url  \
0  https://www.boardgamegeek.com/boardgame/170216   
1  https://www.boardgamegeek.com/boardgame/284378   
2   https://www.boardgamegeek.com/boardgame/31260   
3  https://www.boardgamegeek.com/boardgame/363622   
4  https://www.boardgamegeek.com/boardgame/

In [23]:
missing_price_games = df_out[
    df_out["min_of_min"].isna() & df_out["max_of_max"].isna()
]

print(missing_price_games.head())
print("count:", len(missing_price_games))

            key                                       slug  \
364  Aj58jxUWIj                              glory-to-rome   
435  mGdHNvHD20                        dominion-cornucopia   
445  0z0iXbR2tk                      advanced-squad-leader   
527  -rDOBlRFn7  heroscape-master-set-rise-of-the-valkyrie   
547  iH_gvODLjJ                                     london   

                                          title   bgg_id  \
364                               Glory to Rome  19857.0   
435                        Dominion: Cornucopia  90850.0   
445                       Advanced Squad Leader    243.0   
527  Heroscape Master Set: Rise of the Valkyrie  11170.0   
547                                      London  65781.0   

                                           bgg_url  \
364  https://www.boardgamegeek.com/boardgame/19857   
435  https://www.boardgamegeek.com/boardgame/90850   
445    https://www.boardgamegeek.com/boardgame/243   
527  https://www.boardgamegeek.com/boardgame/11170

In [24]:
def normalize_bgg_id(s):
    return (
        s.astype(str)
         .str.strip()
         .str.replace(r"\.0$", "", regex=True)
    )

rf_pred_df["bgg_id"] = normalize_bgg_id(rf_pred_df["bgg_id"])
df_out["bgg_id"] = normalize_bgg_id(df_out["bgg_id"])

df_merged = rf_pred_df.merge(df_out, on="bgg_id", how="inner")
print(len(df_merged))
df_merged.drop(
    columns=['key', 'slug', 'title', 'bgg_url', 'detail_url', 'n_points'],
    inplace=True,
    errors="ignore"
)

20628


In [25]:
df_merged.to_csv("ridge_predictions_with_price_stats.csv", index=False)

In [26]:
# missing_price_games = df_merged[
#     df_merged["min_of_min"].isna() &
#     df_merged["max_of_max"].isna() &
#     df_merged["mean_of_mean"].isna()
# ]

# print(missing_price_games.head())
# print("count:", len(missing_price_games))

In [27]:
# df = pd.read_parquet(parquet_path)
# print("Original shape:", df.shape)

In [28]:
# df["bgg_id"] = df["bgg_id"].astype(str)
# missing_price_games["bgg_id"] = missing_price_games["bgg_id"].astype(str)

In [29]:
# df_clean = df[~df["bgg_id"].isin(missing_price_games["bgg_id"])]
# print("After removal:", df_clean.shape)

In [30]:
# df_clean.to_parquet(BASE_DIR / "cleaned_data.parquet", index=False)